# Tuning

Find the optimal `n_clusters` and `n_segments` for your data.

Three functions cover different use cases:

| Function | What it does |
|---|---|
| `find_best_combination` | Full grid search — returns best RMSE with complete history |
| `find_pareto_front` | Same grid, filtered to Pareto-optimal points |
| `find_optimal_combination` | Boundary curve for a target data reduction — fastest |

Unlike tsam's built-in tuning, all three evaluate each candidate across **all slices**.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook_connected"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")

## Grid search &amp; RMSE heatmap

In [ ]:
grid = tsam_xarray.find_best_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Tested {len(grid.history)} combinations")
print(f"Best: {grid.n_clusters}c x {grid.n_segments}s (RMSE={grid.rmse:.4f})")

grid.summary_matrix["rmse"].plotly.imshow(
    x="n_segments",
    y="n_clusters",
    title="RMSE by (n_clusters, n_segments)",
)

## Pareto front

`find_pareto_front` runs the same grid search but filters to non-dominated
configurations — those where no other combo has both lower RMSE *and* fewer timesteps.

In [ ]:
pareto = tsam_xarray.find_pareto_front(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    max_timesteps=48,
    show_progress=False,
)
print(f"Pareto-optimal: {len(pareto.history)} of {len(grid.history)} tested")
pareto.summary

## Inspecting the best result

The `best_result` is a full `AggregationResult` — same as from `aggregate()`.

In [ ]:
grid.best_result.cluster_representatives.sel(
    scenario="low",
    variable="solar",
).plotly.line(
    line_shape="hv",
    x="timestep",
    color="cluster",
    facet_col="region",
    title="Best typical periods (solar, low scenario)",
)

## Find optimal for a specific data reduction

`find_optimal_combination` tests only the boundary combos for a target reduction — faster.

In [ ]:
result_opt = tsam_xarray.find_optimal_combination(
    da,
    time_dim="time",
    cluster_dim=["variable", "region"],
    data_reduction=0.05,
    show_progress=False,
)
print(f"Best for 5% reduction: {result_opt.n_clusters}c x {result_opt.n_segments}s")
print(f"RMSE: {result_opt.rmse:.4f}")
result_opt.summary